In [1]:
using LinearAlgebra
using Random
using ProgressMeter
using JLD2
using Ket
using MosekTools
using ppt2
using TensorOperations

ArgumentError: ArgumentError: Package TensorOperations not found in current path.
- Run `import Pkg; Pkg.add("TensorOperations")` to install the TensorOperations package.

In [2]:
forms_4x4 = vcat([jldopen("../pncp_forms_4x4.jld2") do file
    vcat([file[k] for k in keys(file)]...) end]...);

In [3]:
rng = Xoshiro(0)

Xoshiro(0xdb2fa90498613fdf, 0x48d73dc42d195740, 0x8c49bc52dc8a77ea, 0x1911b814c02405e8, 0x22a21880af5dc689)

In [99]:
function rand_vec(dims...; rng=Random.GLOBAL_RNG)
    # return float(rand(rng, -1:1, dims...))
    return randn(rng, dims...)
end

rand_vec (generic function with 1 method)

In [376]:
rand_vec(4, 4)

4×4 Matrix{Float64}:
  1.50851   -0.263823   0.783991   1.34498
  0.636744   0.546708  -0.0706542  0.104208
  1.51526   -0.374519   0.218517   1.59642
 -0.833847   0.3767    -0.332135   0.0388689

In [352]:
function rand_ppt(n::Int, m::Int; rng=Random.GLOBAL_RNG, rand_vec=_rand_vec, ipt=true)
    A = rand_vec(n*m, n*m; rng=rng)
    rho = A * A'
    if ipt
        for i in 1:m, j in i+1:m
            rows = (i - 1) * n + 1:i * n
            cols = (j - 1) * n + 1:j * n
            sym = (rho[rows, cols] + rho[cols, rows]) / 2
            rho[rows, cols] = sym
            rho[cols, rows] = sym
        end
    end
    delta = eigmin(partial_transpose(rho, 1))
    if delta < 0
        return rho - 1. * delta * I
    end
    return rho
end

function rand_ent(n::Int, m::Int; rng=Random.GLOBAL_RNG, rand_vec=_rand_vec)
    A = rand_vec(n*m, n*m, 5*n*n*m*m; rng=rng)
    @tensor rho[i, j] := A[i, k, l] * A[j, k, l]
    return rho / tr(rho)
end

rand_ent (generic function with 1 method)

In [452]:
ppt = rand_ppt(4, 4; rng, rand_vec, ipt=true)
println("eigmin rho: ", eigmin(ppt))
println("eigmin tr.: ", eigmin(partial_transpose(ppt, 1)))
r, w = entanglement_robustness(ppt, [4, 4], 2; solver=Mosek.Optimizer)
println("Robustness: ", r)
println("min trace : ", findmin(tr.(forms_4x4 .* Ref(ppt))))
println("Ampliation: ", findmin(eigmin.(ampliation.(forms_4x4, Ref(ppt), 4, 4))))

eigmin rho: -1.0601420403128337e-14
eigmin tr.: -1.0601420403128337e-14
Robustness: 3.358049297918154e-8
min trace : (96.2435667775108, 1237)
Ampliation: (0.08870785624324501, 650)


In [453]:
ent = rand_ent(4, 4; rng, rand_vec)
println("eigmin rho: ", eigmin(ent))
println("eigmin tr.: ", eigmin(partial_transpose(ent, 1)))
r, w = entanglement_robustness(ent, [4, 4], 2; solver=Mosek.Optimizer)
println("Robustness: ", r)
println("min trace : ", findmin(tr.(forms_4x4 .* Ref(ent))))
println("Ampliation: ", findmin(eigmin.(ampliation.(forms_4x4, Ref(ent), 4, 4))))

eigmin rho: 0.05939903934492634
eigmin tr.: 0.05923598704919736
Robustness: -0.05923598706529322
min trace : (0.5223641398684449, 131)
Ampliation: (0.0009263193232414809, 1500)


In [11]:
function gen_ppt(n::Int, m::Int, forms; n_trials::Int=10000, tol::Float64=1e-6)
    @showprogress for i in 1:n_trials
        ppt = rand_ppt(n, m; rng, rand_vec, pt_sym=true)
        r, idx = findmin(tr.(forms .* Ref(ppt)))
        # e, _ = findmin(eigmin.(ampliation.(Ref(ppt), forms, n, m)))
        e, _ = findmin(eigmin.(ampliation.(forms, Ref(ppt), n, m)))
        if r < -tol
            println("Witness $idx detects entanglement: trace=$r")
            return ppt, forms[idx]
        end
        if e < -tol
            println("Witness $idx detects entanglement: eigmin=$e")
            return ppt, forms[idx]
        end
    end
    return nothing, nothing
end

function gen_ppt_dps(n::Int, m::Int; n_trials::Int=10000, tol::Float64=1e-6)
    @showprogress for i in 1:n_trials
        ppt = rand_ppt(n, m; rng, rand_vec, pt_sym=true)
        r, w = entanglement_robustness(ppt, [n, m], 2; solver=Mosek.Optimizer)
        if r > tol
            println("DPS detects entanglement: $r")
            return ppt, w
        end
    end
    return nothing, nothing
end

gen_ppt_dps (generic function with 1 method)

In [315]:
function rand_psd(dims...; rng=Random.GLOBAL_RNG, rand_vec=_rand_vec)
    A = rand_vec(dims...; rng=rng)
    return A * A'
end

function rand_sep(n::Int, m::Int; rng=Random.GLOBAL_RNG, rand_vec=_rand_vec, n_terms::Int=10)
    rho = zeros(n * m, n * m)
    for _ in 1:n_terms
        A = rand_vec(n, n; rng=rng)
        B = rand_vec(m, m; rng=rng)
        rho += kron(A * A', B * B')
    end
    return rho / tr(rho)
end

rand_sep (generic function with 1 method)

In [ ]:
state = nothing
@showprogress for i in 1:1000
    cand = rand_ent(16, 4; rng, rand_vec)
    r, w = entanglement_robustness(cand, [4, 4], 2; solver=Mosek.Optimizer)
    found = false
    if r > 1e-6
        # println("DPS detects entanglement: $r")
        cand = cand - 1.1 * eigmin(partial_transpose(cand, 1)) * I
        r, w = entanglement_robustness(cand, [4, 4], 2; solver=Mosek.Optimizer)
        if r > 1e-6
            found = true
            println("DPS detects entanglement after adjustment: $r")
            state = cand
            break
        end
        if found
            break
        end
    end
end

Progress: 100%|█████████████████████████████████████████| Time: 0:55:23


In [212]:
ppt, form = gen_ppt_dps(3, 3; n_trials=1000, tol=1e-6)

Progress: 100%|█████████████████████████████████████████| Time: 0:01:06


(nothing, nothing)

In [13]:
# ppt, form = test_ppt2_pncp(4, 4, forms_4x4; n_trials=1000, tol=0.)
# ppt, form = gen_ppt(4, 4, forms_4x4; n_trials=1000, tol=0.)
ppt, form = gen_ppt_dps(4, 4; n_trials=1000, tol=0.)

Progress:  57%|███████████████████████▌                 |  ETA: 0:14:14

InterruptException: InterruptException:

In [ ]:
function test_ppt2_pncp(n::Int, m::Int, forms; n_trials::Int=10000, tol::Float64=1e-6)
    @showprogress for i in 1:n_trials
        ppt = rand_ppt(n, m; rng, rand_vec, pt_sym=true)
        cpp = ampliation(ppt, ppt, n, m)
        r, idx = findmin(tr.(forms .* Ref(cpp)))
        e1, _ = findmin(eigmin.(ampliation.(Ref(cpp), forms, n, m)))
        e2, _ = findmin(eigmin.(ampliation.(forms, Ref(cpp), n, m)))
        if r < -tol
            println("Witness $idx detects entanglement: trace=$r")
            return ppt, forms[idx]
        end
        if e1 < -tol
            println("Witness $idx detects entanglement: eigmin1=$e1")
            return ppt, forms[idx]
        end
        if e2 < -tol
            println("Witness $idx detects entanglement: eigmin2=$e2")
            return ppt, forms[idx]
        end
    end
    return nothing, nothing
end

function test_ppt2_dps(n::Int, m::Int; n_trials::Int=10000, tol::Float64=1e-6)
    @showprogress for i in 1:n_trials
        ppt = rand_ppt(n, m; rng, rand_vec, pt_sym=true)
        cpp = Hermitian(ampliation(ppt, ppt, n, m))
        r, w = entanglement_robustness(cpp, [n, m], 2; solver=Mosek.Optimizer)
        if r > tol
            println("DPS detects entanglement: $r")
            return ppt, w
        end
    end
    return nothing, nothing
end

test_ppt2_dps (generic function with 1 method)

In [4]:
function rand_unitary(n::Int)
    X = (randn(n, n) .+ im * randn(n, n)) ./ sqrt(2)
    F = qr(X)
    Q = Matrix(F.Q)
    R = F.R
    phases = similar(diag(R))
    for i in eachindex(phases)
        r = R[i, i]
        phases[i] = iszero(r) ? 1.0 : r / abs(r)
    end
    return Q * Diagonal(phases)
end

rand_unitary (generic function with 1 method)

In [5]:
function rand_ent_choi(dim_in::Int=2, dim_out::Int=2)
    d_total = dim_in * dim_out

    # random unitary on the total space
    U = rand_unitary(d_total)

    # construct maximally entangled vector (match Python's row-major flattening)
    phi_plus = zeros(ComplexF64, d_total)
    for i in 1:min(dim_in, dim_out)
        idx = (i - 1) * dim_out + i
        phi_plus[idx] = 1
    end
    phi_plus ./= sqrt(dim_in)

    # apply unitary
    psi = U * phi_plus

    # pure state projector
    rho = psi * psi'

    # interpret as Choi and normalize trace to dim_in
    choi = rho .* (dim_in / tr(rho))

    return choi
end

rand_ent_choi (generic function with 3 methods)

In [15]:
A = rand_ent_choi(4, 4)
eigmin(A), eigmin(partial_transpose(A, 1))

(-2.3525475966724157e-16, -1.6506286614878025)